# SNOMED CT — API & bulk

SNOMED CT is the canonical clinical terminology — diagnoses,
procedures, findings, body sites, etc. `biodb.snomed` covers both
modes, but the bulk path is **deliberately parser-only** because of
SNOMED CT's licensing posture:

| Mode | Functions |
|---|---|
| **API (OLS4 REST)** | `query_concept`, `search_concepts`, `get_descendants`, `get_ancestors`, `get_children`, `get_parents` |
| **Bulk (local parser)** | `load_concept_csv`, `load_concept_csv_from_zip` — parses what *you* obtained from Athena |

Per-concept lookups route through EBI's [OLS4](https://www.ebi.ac.uk/ols4/)
(~376 k SNOMED terms indexed) and accept the bare SNOMED ID
(`38341003`), the CURIE (`SNOMED:38341003`), or the full IRI
(`http://snomed.info/id/38341003`). The bulk parser reads the CONCEPT.csv
you downloaded yourself from [OHDSI Athena](https://athena.ohdsi.org)
after accepting the SNOMED CT license — bioDB does not host or
redistribute SNOMED CT bytes.

In [1]:
from biodb import snomed

## 1. API mode — per-concept lookups via OLS4

No local data required. Pass an `int`, a CURIE, or an IRI — they all
normalise to the same canonical record.

In [2]:
record = snomed.query_concept(38341003)
{
    "obo_id": record["obo_id"],
    "label": record["label"],
    "iri": record["iri"],
    "synonyms": record["synonyms"][:3] if record["synonyms"] else [],
    "is_obsolete": record["is_obsolete"],
}

{'obo_id': 'SNOMED:38341003',
 'label': 'Hypertensive disorder',
 'iri': 'http://snomed.info/id/38341003',
 'synonyms': [],
 'is_obsolete': False}

Full-text search across SNOMED concept labels / synonyms /
definitions. Solr-backed — fuzzy by default, pass `exact=True` to
require an exact match.

In [3]:
hits = snomed.search_concepts("hypertension", rows=5)
hits[["obo_id", "label"]]

,obo_id,label
0,SNOMED:10725009,Benign hypertension
1,SNOMED:123799005,Renovascular hypertension
2,SNOMED:123800009,Goldblatt hypertension
3,SNOMED:1356877007,Stable hypertension
4,SNOMED:162659009,Hypertension resolved


Walk the SNOMED IS-A graph — ancestors (up to the SCT root) and
descendants (the full subtree). Both return DataFrames; pagination
is handled internally.

In [4]:
parents = snomed.get_parents(38341003)
print(f"Hypertensive disorder has {len(parents)} direct parents:")
parents[["obo_id", "label"]]

Hypertensive disorder has 2 direct parents:


,obo_id,label
0,SNOMED:49601007,Disorder of cardiovascular system
1,SNOMED:366157005,Cardiovascular measurement - finding


In [5]:
descendants = snomed.get_descendants(38341003)
print(f"Hypertensive disorder has {len(descendants)} transitive descendants:")
descendants[["obo_id", "label"]].head(8)

Hypertensive disorder has 128 transitive descendants:


,obo_id,label
0,SNOMED:845891000000103,Hypertension resistant to drug therapy
1,SNOMED:84094009,Rebound hypertension
2,SNOMED:720568003,Brachydactyly and arterial hypertension syndrome
3,SNOMED:71701000119105,Hypertension in chronic kidney disease due to ...
4,SNOMED:712832005,Supine hypertension
5,SNOMED:71421000119105,Hypertension in chronic kidney disease due to ...
6,SNOMED:706882009,Hypertensive crisis
7,SNOMED:704667004,Hypertension concurrent and due to end stage r...


## 2. Bulk mode — local CONCEPT.csv parser

**bioDB does not host the SNOMED CT bytes.** SNOMED CT is a licensed
vocabulary (free in Member countries via UMLS/IHTSDO; paid Affiliate
license elsewhere), and onward redistribution from a public mirror
isn't permitted. So the bulk path is *bring-your-own-file*.

### How to get a CONCEPT.csv

1. Register a free account at <https://athena.ohdsi.org>.
2. In your profile, **accept the SNOMED CT license** (and any other
   vocabulary licenses you'll use — RxNorm, CPT, ICD, …).
3. On the Vocabularies page, select the vocabularies you want and
   click **Download**. Athena builds the bundle asynchronously and
   emails you a download link (minutes to ~1 hr depending on size).
4. Save the resulting `.zip` somewhere — bioDB reads it in place,
   no need to unzip first.

In [6]:
print(f"Download page: {snomed.ATHENA_DOWNLOAD_PAGE}")
print(f"Suggested cache: {snomed.CACHE_DIR}")

Download page: https://athena.ohdsi.org/vocabulary/list
Suggested cache: /Users/bschilder/.cache/biodb/snomed


### Parse a local `CONCEPT.csv`

`load_concept_csv` returns a DataFrame with the OHDSI CDM schema.
Pass `vocabulary_id="SNOMED"` to keep only the SNOMED slice — the
bundle mixes ~10 vocabularies into one file.

We demonstrate against a tiny synthetic CSV; the real Athena bundle
(~175 MB) parses with the same call.

In [7]:
import tempfile
from pathlib import Path

_tmp = Path(tempfile.mkdtemp())
fixture = _tmp / "CONCEPT.csv"
fixture.write_bytes(
    b"concept_id\tconcept_name\tdomain_id\tvocabulary_id\tconcept_class_id\t"
    b"standard_concept\tconcept_code\tvalid_start_date\tvalid_end_date\tinvalid_reason\n"
    b"12345\tHypertensive disorder\tCondition\tSNOMED\tClinical Finding\tS\t"
    b"38341003\t1970-01-01\t2099-12-31\t\n"
    b"67890\tType 2 diabetes mellitus\tCondition\tSNOMED\tClinical Finding\tS\t"
    b"44054006\t1970-01-01\t2099-12-31\t\n"
    b"1112807\tAspirin 81 MG\tDrug\tRxNorm\tBranded Drug\tS\t315431\t1970-01-01\t"
    b"2099-12-31\t\n"
)

df = snomed.load_concept_csv(fixture, vocabulary_id="SNOMED")
df[["concept_id", "concept_name", "vocabulary_id", "concept_code"]]

,concept_id,concept_name,vocabulary_id,concept_code
0,12345,Hypertensive disorder,SNOMED,38341003
1,67890,Type 2 diabetes mellitus,SNOMED,44054006


### Parse straight from the Athena bundle `.zip`

`load_concept_csv_from_zip` extracts a member in-memory — no need
to unzip first. Match is on basename, so the function handles both
flat zips and zips wrapped under a single sub-directory.

In [8]:
import zipfile

zip_path = _tmp / "vocabulary_download_v5_demo.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    zf.writestr("CONCEPT.csv", fixture.read_bytes())
    zf.writestr("CONCEPT_RELATIONSHIP.csv", "concept_id_1\tconcept_id_2\n1\t2\n")

df = snomed.load_concept_csv_from_zip(zip_path)
df[["concept_id", "concept_name", "vocabulary_id"]]

,concept_id,concept_name,vocabulary_id
0,12345,Hypertensive disorder,SNOMED
1,67890,Type 2 diabetes mellitus,SNOMED
2,1112807,Aspirin 81 MG,RxNorm


### Schema

The columns map 1-to-1 to the OHDSI CDM concept table — see
<https://ohdsi.github.io/CommonDataModel/cdm54.html#CONCEPT>.

In [9]:
expected_columns = [
    "concept_id",  # integer primary key
    "concept_name",
    "domain_id",  # Condition / Drug / Procedure / Measurement / Device / Observation / …
    "vocabulary_id",  # SNOMED, RxNorm, LOINC, …
    "concept_class_id",
    "standard_concept",  # 'S' = standard, 'C' = classification, NaN = non-standard
    "concept_code",  # the original SNOMED ID (e.g. 38341003 for hypertensive disorder)
    "valid_start_date",
    "valid_end_date",
    "invalid_reason",
]
for col in expected_columns:
    print(col)

concept_id
concept_name
domain_id
vocabulary_id
concept_class_id
standard_concept
concept_code
valid_start_date
valid_end_date
invalid_reason
